# NER on articles_nite.json + Social Network Graph Data

1. Load articles from `articles_nite.json` (local file, no DB).
2. Run NER with `Jean-Baptiste/roberta-large-ner-english`.
3. Save all entities to `ner_results.csv`.
4. Build PER-based social network: **nodes** (person + mention count) and **edges** (co-occurrence weight).
5. Save `network_nodes.csv` and `network_edges.csv`.

In [1]:
import json
import logging
import re
import csv

import torch
import pandas as pd

from collections import Counter
from itertools import combinations
from pathlib import Path
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

C:\Users\johna\miniforge3\envs\textmining2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────

DATA_DIR    = Path(r"D:\development\2026_python\2026_thecall_public\thecall\data")
INPUT_JSON  = DATA_DIR / 'articles_nite.json'

# Output files
NER_CSV          = DATA_DIR / 'ner_results.csv'
NODES_CSV        = DATA_DIR / 'network_nodes.csv'
EDGES_CSV        = DATA_DIR / 'network_edges.csv'

MODEL_NAME  = 'Jean-Baptiste/roberta-large-ner-english'
BATCH_SIZE  = 32
AGGREGATION = 'simple'

In [4]:
# ── Device setup ──────────────────────────────────────────────────────────────
# Priority: CUDA (Nvidia) > XPU (Intel Arc/Data Center) > CPU
if torch.cuda.is_available():
    DEVICE = 0
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif hasattr(torch, 'xpu') and torch.xpu.is_available():
    DEVICE = 'xpu'
    DEVICE_NAME = 'Intel XPU'
else:
    DEVICE = -1
    DEVICE_NAME = 'CPU'

print(f'PyTorch version : {torch.__version__}')
print(f'Selected device : {DEVICE_NAME}')

PyTorch version : 2.7.0+xpu
Selected device : Intel XPU


In [5]:
# ── Model loading ─────────────────────────────────────────────────────────────
logging.info(f'Loading model: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model     = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
model.eval()

if DEVICE == 'xpu':
    import intel_extension_for_pytorch as ipex
    model = model.to('xpu')
    model = ipex.optimize(model, dtype=torch.float32)
    pipeline_device = 'xpu'
    print(f'Intel PyTorch Extension version: {ipex.__version__}')
else:
    pipeline_device = DEVICE

ner_pipe = pipeline(
    'ner',
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy=AGGREGATION,
    device=pipeline_device,
)

logging.info('Pipeline ready.')

INFO: Loading model: Jean-Baptiste/roberta-large-ner-english
C:\Users\johna\miniforge3\envs\textmining2\Lib\site-packages\intel_extension_for_pytorch\nn\utils\_weight_prepack.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-05-01 08:54:19,460 - _logger.py - IPEX - INFO - Currently split master weight for xpu only support sgd
2026-05-01 08:54:19,468 - _logger.py - IPEX - INFO - Conv BatchNorm folding failed during the optimize process.
2026-05-01 08:54:19,477 - _logger.py - IPEX - INFO - Linear BatchNorm folding failed during the optimize process.
Device set to use xpu
INFO: Pipeline ready.


Intel PyTorch Extension version: 2.7.10+xpu


In [6]:
# ── Load articles from JSON ───────────────────────────────────────────────────
with open(INPUT_JSON, 'r', encoding='utf-8') as f:
    articles = json.load(f)

print(f'Loaded {len(articles)} articles from {INPUT_JSON.name}')

# Build (id, text) tuples — same shape the old DB query returned
all_data = [
    (a['id_articles_nite'], a.get('text', ''))
    for a in articles
    if a.get('text')   # skip articles with no text
]
print(f'{len(all_data)} articles have text for NER')

Loaded 441 articles from articles_nite.json
440 articles have text for NER


In [7]:
# ── Helper functions (unchanged from original) ────────────────────────────────

def is_valid_entity(text):
    """Checks if the entity is substantial enough to be kept."""
    if re.match(r'^[.\s\d\W_]+$', text):
        return False
    if len(text) < 2:
        return False
    if len(text) == 2 and text.upper() not in ['KS', 'MO']:
        return False
    return True


def clean_entity_text(text):
    """Cleans OCR artifacts, normalises to Title Case, removes leading 'The'."""
    text = re.sub(r'^[^\w]+|[^\w]+$', '', text).strip()
    text = text.title()
    if text.startswith('The '):
        text = text[4:].strip()
    return text


def chunk_text(text, max_chars=3000):
    """Splits text into chunks to fit within RoBERTa's token limits."""
    return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]


def strip_p_tags(html_text):
    """Convert <p>...</p> blocks into plain prose for NER."""
    text = re.sub(r'</?p>', ' ', html_text or '')
    return re.sub(r'\s+', ' ', text).strip()

In [8]:
# ── NER task (adapted: text field is already the <p>-wrapped string) ───────────

def ner_task(all_data, ner_pipe):
    results = []
    logging.info(f'Starting NER on {len(all_data)} articles...')

    for article_id, raw_text in tqdm(all_data, desc='Processing Articles'):
        article_text = strip_p_tags(raw_text)
        if not article_text:
            logging.warning(f'id={article_id}: text is empty, skipping.')
            continue

        for chunk in chunk_text(article_text):
            try:
                entities = ner_pipe(chunk)
                for ent in entities:
                    cleaned_val = clean_entity_text(ent['word'])
                    if is_valid_entity(cleaned_val):
                        results.append({
                            'entity_type':      ent['entity_group'],
                            'entity_value':     cleaned_val,
                            'id_articles_nite': article_id,
                            'score':            round(float(ent['score']), 4),
                        })
            except Exception as e:
                logging.error(f'NER failed for id={article_id}: {e}')

    return results

In [9]:
# ── Run NER ───────────────────────────────────────────────────────────────────
ner_results = ner_task(all_data, ner_pipe)
print(f'Total entities extracted: {len(ner_results)}')

INFO: Starting NER on 440 articles...
Processing Articles: 100%|███████████████████████████████████████████████████████████| 440/440 [00:35<00:00, 12.39it/s]

Total entities extracted: 17006


In [10]:
# ── Save NER results to CSV ───────────────────────────────────────────────────
df_ner = pd.DataFrame(ner_results)
df_ner.to_csv(NER_CSV, index=False, encoding='utf-8')
print(f'Saved {len(df_ner)} entities to {NER_CSV.name}')

# Quick summary
print(f"\nEntity type counts:")
print(df_ner['entity_type'].value_counts().to_string())

Saved 17006 entities to ner_results.csv

Entity type counts:
entity_type
PER     6944
ORG     6868
LOC     1712
MISC    1482


In [11]:
# ── Preview NER output ────────────────────────────────────────────────────────
df_ner.head(10)

,entity_type,entity_value,id_articles_nite,score
0,PER,Jim,1,0.7992
1,PER,Grand Dad,1,0.9936
2,PER,Nunnelly,1,0.9996
3,PER,Charles Allen,1,0.9998
4,ORG,Chiefs,1,0.9983
5,ORG,Chiefs,1,0.9972
6,PER,Paul Booker,1,0.9999
7,ORG,Chiefs,1,0.9985
8,PER,Carl Clark,1,0.9999
9,ORG,Chiefs,1,0.9983


## Social Network Graph Data

Build two files from the PER entities:

- **`network_nodes.csv`**: one row per unique person, with `mention_count` (total times they appear across all articles) so node size can be proportional.
- **`network_edges.csv`**: one row per person-pair that co-occur in at least one article, with `co_occurrence_count` (number of articles they share) so edge weight/thickness can be proportional.

In [12]:
# ── Build social network from PER entities ─────────────────────────────────────

# Filter to PER only
df_per = df_ner[df_ner['entity_type'] == 'PER'].copy()
print(f'PER entities: {len(df_per)}')

# ── NODES: total mention count per person ──────────────────────────────────────
node_counts = df_per['entity_value'].value_counts().reset_index()
node_counts.columns = ['person', 'mention_count']
node_counts = node_counts.sort_values('mention_count', ascending=False).reset_index(drop=True)

print(f'Unique persons (nodes): {len(node_counts)}')
print(f'\nTop 15 most-mentioned:')
print(node_counts.head(15).to_string(index=False))

PER entities: 6944
Unique persons (nodes): 1682

Top 15 most-mentioned:
          person  mention_count
    Jim Nunnelly            263
      Carl Clark            201
             Ike            182
         Ed West            181
     Ike Gardner            180
     Paul Booker            170
   Charles Allen            156
        Nunnelly            154
   Ollie Hubbard            153
             Jim            137
Lloyd Greenfield            129
          Booker            100
       Grand Dad             86
   Bubba Cleaver             75
           Allen             71


In [13]:
# ── EDGES: co-occurrence count (articles shared by each pair) ──────────────────

# For each article, get the unique set of PER names
per_by_article = (
    df_per.groupby('id_articles_nite')['entity_value']
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

# Count co-occurrences: for every article with 2+ people,
# generate all unordered pairs and tally them.
edge_counter = Counter()
for article_id, persons in per_by_article.items():
    if len(persons) < 2:
        continue
    for pair in combinations(persons, 2):
        edge_counter[pair] += 1

# Build DataFrame
edges = [
    {'source': p1, 'target': p2, 'co_occurrence_count': count}
    for (p1, p2), count in edge_counter.items()
]
df_edges = pd.DataFrame(edges).sort_values('co_occurrence_count', ascending=False).reset_index(drop=True)

print(f'Unique edges (person pairs): {len(df_edges)}')
print(f'\nTop 15 strongest connections:')
print(df_edges.head(15).to_string(index=False))

Unique edges (person pairs): 27059

Top 15 strongest connections:
       source           target  co_occurrence_count
   Carl Clark      Paul Booker                  122
   Carl Clark          Ed West                  118
   Carl Clark      Ike Gardner                  117
   Carl Clark    Charles Allen                  116
   Carl Clark     Jim Nunnelly                  111
   Carl Clark    Ollie Hubbard                  110
Charles Allen      Paul Booker                  108
Ollie Hubbard      Paul Booker                  107
      Ed West      Paul Booker                  106
   Carl Clark Lloyd Greenfield                  102
      Ed West    Ollie Hubbard                  101
  Ike Gardner      Paul Booker                  100
      Ed West     Jim Nunnelly                  100
Charles Allen          Ed West                   99
 Jim Nunnelly      Paul Booker                   99


In [14]:
# ── Save network files ────────────────────────────────────────────────────────
node_counts.to_csv(NODES_CSV, index=False, encoding='utf-8')
df_edges.to_csv(EDGES_CSV, index=False, encoding='utf-8')

print(f'Saved {len(node_counts)} nodes to {NODES_CSV.name}')
print(f'Saved {len(df_edges)} edges to {EDGES_CSV.name}')

Saved 1682 nodes to network_nodes.csv
Saved 27059 edges to network_edges.csv


In [15]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  OUTPUT FILES')
print('=' * 60)
print(f'  NER results:    {NER_CSV}')
print(f'  Network nodes:  {NODES_CSV}')
print(f'  Network edges:  {EDGES_CSV}')
print('=' * 60)
print(f'  Articles processed:     {len(all_data)}')
print(f'  Total entities:         {len(df_ner)}')
print(f'  PER entities:           {len(df_per)}')
print(f'  Unique persons (nodes): {len(node_counts)}')
print(f'  Person pairs (edges):   {len(df_edges)}')
print('=' * 60)


  OUTPUT FILES
  NER results:    D:\development\2026_python\2026_thecall_public\thecall\data\ner_results.csv
  Network nodes:  D:\development\2026_python\2026_thecall_public\thecall\data\network_nodes.csv
  Network edges:  D:\development\2026_python\2026_thecall_public\thecall\data\network_edges.csv
  Articles processed:     440
  Total entities:         17006
  PER entities:           6944
  Unique persons (nodes): 1682
  Person pairs (edges):   27059
